In [91]:
import pandas as pd
import numpy as np
import getpass
import warnings
# from spacetrack import SpaceTrackClient

# Ignorar os avisos de tipos de dados
warnings.filterwarnings('ignore')

## Read files

In [ ]:
print("\n--- DOWNLOAD FROM CELESTRAK ---")
df_cel = pd.read_csv("https://celestrak.org/pub/satcat.csv")

print("\n--- DOWNLOAD GCAT (Jonathan McDowell) ---")
df_jon = pd.read_csv("https://planet4589.org/space/gcat/tsv/cat/satcat.tsv", sep='\t', low_memory=False, skipinitialspace=True, skiprows=[1])

In [93]:
path = "../DATASETS_SATTELITES/"
#df_cel = pd.read_csv(path + "celestrack/celestrak_data_api.csv")
#df_jon = pd.read_csv(path + "jonathan/gcat_data.csv")

In [94]:
df_jon_backup = df_jon

### Jonathan's type erros:

- there are some cases where the NORAD CAT is wrong
- We noticed that #JCAT and NORAD CAT have similar counting
    - however there are some values in NORAD columns saved as "NNA"
    - We verified with celestrack, the missing values, are indeed equal as in JCAT value
    - So we corrected those missing values

    Example:
    - S40898,40898,2015-038
    - S40899,NNA,  2015-049

- Also, there are some typos:
    - jcats_to_fix = ['S63360', 'S66550', 'S67286', 'S67487', 'S67488', 'S67489', 'S67490', 'S67491', 'S67492' ]
    - Example:
        - S63359,63359
        - S63360,63359 <------------ error, should be 63360
        - S63361,63361


In [95]:
# Save the initial number of columns to ensure we don't add any
initial_col_count = len(df_jon.columns)

# List of the specific cases that we know are Jonathan's typos
jcats_to_fix = [
    'S63360', 'S66550', 'S67286', 
    'S67487', 'S67488', 'S67489', 
    'S67490', 'S67491', 'S67492'
]

# Clean spaces and create conditions (masks) to know which rows to modify
is_nna = df_jon['Satcat'].astype(str).str.strip() == 'NNA'
is_error = df_jon['#JCAT'].astype(str).str.strip().isin(jcats_to_fix)

# Combine both conditions: we want to modify rows that are NNA "OR" (|) that are an Error
rows_to_modify = is_nna | is_error

# We access the 'Satcat' column directly (only in the rows identified above) 
# and replace it with the value from the '#JCAT' column (removing the "S" to keep only the number)
df_jon.loc[rows_to_modify, 'Satcat'] = df_jon.loc[rows_to_modify, '#JCAT'].astype(str).str.replace(r'[^0-9]', '', regex=True)

# Final checks
print(f"Initial columns: {initial_col_count}")
print(f"Final columns: {len(df_jon.columns)} (No columns added!)")

print("\n--- Proof of corrected errors ---")
# Let's show the errors (using the is_error mask) to confirm the Satcat was updated
print(df_jon.loc[is_error, ['#JCAT', 'Satcat', 'Name']])

print("\n--- Proof of filled NNAs (first 5) ---")
# Let's show the former NNAs to confirm
print(df_jon.loc[is_nna, ['#JCAT', 'Satcat', 'Name']].head())

# Save the clean file, ready to be merged
df_jon.to_csv('gcat_data_preprocessed.csv', index=False)
print("\nFile saved as 'gcat_data_preprocessed.csv'")

Initial columns: 42
Final columns: 42 (No columns added!)

--- Proof of corrected errors ---
        #JCAT Satcat                    Name
63359  S63360  63360  Electron 63 Kick Stage
66549  S66550  66550       CZ-7A Y10 Stage 3
67285  S67286  67286                 Mule-4T
67486  S67487  67487                 USA 575
67487  S67488  67488                 USA 576
67488  S67489  67489                 USA 577
67489  S67490  67490                 USA 578
67490  S67491  67491                 USA 579
67491  S67492  67492                 USA 580

--- Proof of filled NNAs (first 5) ---
Empty DataFrame
Columns: [#JCAT, Satcat, Name]
Index: []

File saved as 'gcat_data_preprocessed.csv'


## Compare Datasets

Without the previous clean up, celestrack had 623 different NORAD values, but that happened because of values "NNA" that are now corrected.

Now we onl have 16 different values in Celestrak that are not in Jonathan's

In [96]:

# 1. Load the CSV files

# 2. Extract the columns and convert to integers (this resolves leading zeros)
# pd.to_numeric with errors='coerce' converts invalid/text values to NaN, which we then remove (dropna)
gcat_ids = set(pd.to_numeric(df_jon['Satcat'], errors='coerce').dropna().astype(int))
celestrak_ids = set(pd.to_numeric(df_cel['NORAD_CAT_ID'], errors='coerce').dropna().astype(int))

# 3. Calculate the differences using Set operations
in_celestrak_not_gcat = celestrak_ids - gcat_ids
in_gcat_not_celestrak = gcat_ids - celestrak_ids

# 4. Print the results
print(f"Total IDs in GCAT: {len(gcat_ids)}")
print(f"Total IDs in Celestrak: {len(celestrak_ids)}")
print(f"IDs in Celestrak missing from GCAT: {len(in_celestrak_not_gcat)}")
print(f"IDs in GCAT missing from Celestrak: {len(in_gcat_not_celestrak)}")

# Show some examples of IDs missing from GCAT
if len(in_celestrak_not_gcat) > 0:
    print(f"Examples (in Celestrak but not GCAT): {list(in_celestrak_not_gcat)}")

Total IDs in GCAT: 68196
Total IDs in Celestrak: 68088
IDs in Celestrak missing from GCAT: 1
IDs in GCAT missing from Celestrak: 109
Examples (in Celestrak but not GCAT): [68261]


In [97]:
(df_jon['Status'].str.strip() == '-').sum()
# df_cel['OPS_STATUS_CODE'].unique()
(df_cel['OPS_STATUS_CODE'].str.strip() == '-').sum()

np.int64(1459)

We noticed there are id's in celestrack that aren't in Jonathan's report

We must merge the missing id's

In [98]:
# Create a temporary column to compare clean integer IDs
df_jon['clean_id'] = pd.to_numeric(df_jon['Satcat'], errors='coerce').astype('Int64')
df_cel['clean_id'] = pd.to_numeric(df_cel['NORAD_CAT_ID'], errors='coerce').astype('Int64')

# Get the valid IDs from df_jon
jon_ids = set(df_jon['clean_id'].dropna().astype(int))

# Filter rows in df_cel that are missing from df_jon
missing_in_jon = df_cel[
    df_cel['clean_id'].notna() & 
    ~df_cel['clean_id'].isin(jon_ids)
].copy()

# Dictionaries to translate Celestrak categorical data to GCAT format
type_mapping = {
    'PAY': 'P',
    'R/B': 'R',
    'DEB': 'D',
    'UNK': 'U'
}

owner_mapping = {
    'CIS': 'RU',
    'PRC': 'CN',
    'FR':  'F',
    'JPN': 'J',
    'IND': 'IN',
    'ESA': 'I-ESA',
    'TBD': '-'
}

# New mapping for the Operational Status
# +: Operational -> O
# -: Nonoperational -> R
# P: Partially Operational -> O
# B: Backup/Standby -> AR
# S: Spare -> AR
# X: Extended Mission -> O
# D: Decayed -> D
status_mapping = {
    '+': 'O',
    '-': 'R',
    'P': 'O',
    'B': 'AR',
    'S': 'AR',
    'X': 'O',
    'D': 'R'
}

# Function to safely convert dates from YYYY-MM-DD to YYYY Mmm D
def format_date(date_series):
    return pd.to_datetime(date_series, errors='coerce').dt.strftime('%Y %b %e')

# Map direct columns
missing_in_jon['Satcat'] = missing_in_jon['NORAD_CAT_ID']
missing_in_jon['Name'] = missing_in_jon['OBJECT_NAME']
missing_in_jon['Piece'] = missing_in_jon['OBJECT_ID']
missing_in_jon['Apogee'] = missing_in_jon['APOGEE']
missing_in_jon['Perigee'] = missing_in_jon['PERIGEE']
missing_in_jon['Inc'] = missing_in_jon['INCLINATION']

# Apply dictionaries to transform categorical values
missing_in_jon['Type'] = missing_in_jon['OBJECT_TYPE'].map(type_mapping).fillna(missing_in_jon['OBJECT_TYPE'])
missing_in_jon['State'] = missing_in_jon['OWNER'].map(owner_mapping).fillna(missing_in_jon['OWNER'])

# Clean up possible blank spaces in Celestrak statuses before mapping
clean_status = missing_in_jon['OPS_STATUS_CODE'].str.strip()
missing_in_jon['Status'] = clean_status.map(status_mapping).fillna(clean_status)

# Transform dates
missing_in_jon['LDate'] = format_date(missing_in_jon['LAUNCH_DATE'])
missing_in_jon['DDate'] = format_date(missing_in_jon['DECAY_DATE'])

# Store original columns order
jon_cols = df_jon.columns.drop('clean_id').tolist()

# Drop the helper 'clean_id'
df_jon = df_jon.drop(columns=['clean_id'])

# Keep only the columns that exist in df_jon, matching the order exactly
mapped_missing = missing_in_jon.reindex(columns=jon_cols)

# Combine the original dataset with the newly mapped records
combined_df = pd.concat([df_jon, mapped_missing], ignore_index=True)

# Print the summary results
print(f"Original GCAT rows: {len(df_jon)}")
print(f"Rows mapped and added from Celestrak: {len(mapped_missing)}")
print(f"Total rows in new combined DataFrame: {len(combined_df)}")

Original GCAT rows: 68260
Rows mapped and added from Celestrak: 1
Total rows in new combined DataFrame: 68261


In [99]:
# Example added to GCAT from Clestrak
combined_df[combined_df['Satcat'] == 68261]['LDate']

68260    2025 Oct 26
Name: LDate, dtype: str

### Drop Unnecessary columns

We chose to not use all data available, for it is not relevant for our visualization project

In [100]:
# List of columns to drop from the merged dataframe
cols_to_drop = [
    'Launch_Tag', 'PLName', 'Dest', 'Bus', 'Motor', 
    'MassFlag', 'DryMass', 'DryFlag', 'TotMass', 'TotFlag', 
    'LFlag', 'DFlag', 'SpanFlag', 'ODate', 'Perigee', 'PF', 
    'Apogee', 'AF', 'Inc', 'IF', 'OpOrbit', 'OQUAL', 'AltNames'
]

# Drop the columns from the merged dataframe
# Using errors='ignore' ensures the script doesn't crash if a column is already missing
combined_df = combined_df.drop(columns=cols_to_drop, errors='ignore')

# Print the remaining columns to verify
print("\n--- Remaining columns after drop ---")
print(combined_df.columns.tolist())


--- Remaining columns after drop ---
['#JCAT', 'Satcat', 'Piece', 'Type', 'Name', 'LDate', 'Parent', 'SDate', 'Primary', 'DDate', 'Status', 'Owner', 'State', 'Manufacturer', 'Mass', 'Length', 'Diameter', 'Span', 'Shape']


### Add Space Station and Analyst

Jonathan only has "P" (Payload), "R/B" (Rocket Body) and "D" (Debris)

But we have data from celestrak about space stations and Analyst (Objects unkown)

In [101]:
# Function to clean the messy Type strings based on GCAT official classification
def clean_type(t):
    t_clean = str(t).strip().upper()
    if pd.isna(t) or t_clean == 'NAN':
        return 'Unknown'
    
    if t_clean.startswith('P'):
        return 'Satellite'
    elif t_clean.startswith('C'):
        return 'Component'
    elif t_clean.startswith('R'):
        return 'Rocket Body'
    elif t_clean.startswith('D'):
        return 'Debris'
    elif t_clean.startswith('S'):
        return 'Suborbital'
    elif t_clean.startswith('X'):
        return 'Removed'
    elif t_clean.startswith('Z'):
        return 'Ghost Entry'
    else:
        return 'Unknown'

combined_df['Type'] = combined_df['Type'].apply(clean_type)

# =====================================================================
# SECOND MERGE: ADD SPACE STATIONS AND ANALYSTS
# =====================================================================

# Load the extra files and merge them immediately into one temporary structure
df_temp = pd.concat([
    pd.read_csv(path + 'celestrack/Space_Station/space_station.csv', low_memory=False).assign(Type='Space Station'),
    pd.read_csv(path + 'celestrack/Analyst/analyst_sattelites.csv', low_memory=False).assign(Type='In Analysis')
], ignore_index=True)

# Align ID column and clean for matching
df_temp['Satcat'] = df_temp['NORAD_CAT_ID'].astype(str).str.strip()
combined_df['Satcat'] = combined_df['Satcat'].astype(str).str.strip()

# Update Type for objects that already exist in combined_df
existing_mask = df_temp['Satcat'].isin(combined_df['Satcat'])
type_map = df_temp[existing_mask].set_index('Satcat')['Type'].to_dict()
combined_df['Type'] = combined_df['Satcat'].map(type_map).fillna(combined_df['Type'])

# Append only the completely new objects directly to combined_df
new_objects = df_temp[~existing_mask].copy()

# Filter columns and reindex to match combined_df before concat
main_columns = combined_df.columns.tolist()
new_objects = new_objects.reindex(columns=main_columns)

# Overwrite combined_df with the appended rows
combined_df = pd.concat([combined_df, new_objects], ignore_index=True)

# Cleanup temporary dataframe to free memory
del df_temp

# Check operational satellites (Status O and Type Satellite)
op_sat_count = combined_df[(combined_df['Status'].str.strip() == 'O') & (combined_df['Type'] == 'Satellite')].shape[0]

print("\n=== SCRIPT SUMMARY ===")
print(f"Operational Satellites: {op_sat_count}")
print(f"Total rows in dataset: {len(combined_df)}")


=== SCRIPT SUMMARY ===
Operational Satellites: 17628
Total rows in dataset: 68705


### Add Operational Status from celestrak

In [102]:
#####  STATUS OP ################

# make sure they aren't float and int, but both int
df_cel['NORAD_CAT_ID'] = pd.to_numeric(df_cel['NORAD_CAT_ID'], errors='coerce').astype('Int64')
combined_df['Satcat'] = pd.to_numeric(combined_df['Satcat'], errors='coerce').astype('Int64')

# 1. Criar um mapeamento (Dicionário) baseado no ID do Celestrak
# O index é o NORAD_CAT_ID e o valor é o OPS_STATUS_CODE
celestrak_lookup = df_cel.set_index('NORAD_CAT_ID')['OPS_STATUS_CODE']

# 2. Adicionar a coluna ao combined_df através do mapeamento pelo 'Satcat'
# O pandas procura o ID do Satcat no index do Celestrak e traz o respetivo status
combined_df['OPS_STATUS_CODE'] = combined_df['Satcat'].map(celestrak_lookup)

##### LAUNCH SITE #######################

# 4. Criar o dicionário de mapeamento para o local de lançamento
site_lookup = df_cel.set_index('NORAD_CAT_ID')['LAUNCH_SITE']

# 5. Adicionar a nova coluna ao combined_df
combined_df['LAUNCH_SITE'] = combined_df['Satcat'].map(site_lookup)

# 6. Verificar o resultado final
colunas_interesse = ['Satcat', 'Name', 'Status', 'OPS_STATUS_CODE', 'LAUNCH_SITE']
print(combined_df[44850:44860][colunas_interesse])

       Satcat                 Name Status OPS_STATUS_CODE LAUNCH_SITE
44850   44851    Fregat No. 112-10      O             NaN       PLMSC
44851   44852               NANOVA      R               D       SRILR
44852   44853           Tyvak-0129      R               D       SRILR
44853   44854           Duchifat-3      R               D       SRILR
44854   44855   Lemur-2-JPGSquared      R               D       SRILR
44855   44856              Izanagi      R               D       SRILR
44856   44857           RISAT-2BR1      O               +       SRILR
44857   44858        PSLV adapter?      R               D       SRILR
44858   44859           1HOPSAT-TD      O               +       SRILR
44859   44860  Lemur-2-HiMomAndDad      R               D       SRILR


### Add Metadata

Check how many operational sattelite there are. Because we have a dataset with metadata for 14167 satellites

In [103]:
# Filter only rows where Type is 'Satellite' and Status is 'O'
operational_satellites = combined_df[(combined_df['Type'] == 'Satellite') & (combined_df['Status'].str.strip() == 'O')]

# Count the results
total_op_sat = operational_satellites.shape[0]

print(f"Total Operational Satellites: {total_op_sat}")

# Optional: See the top owners of these operational satellites
print("\nTop 5 Owners of Operational Satellites:")
print(operational_satellites['Owner'].value_counts().head(5))

Total Operational Satellites: 17628

Top 5 Owners of Operational Satellites:
Owner
SPXS       10049
ONEWEBN      524
NROC         265
KUIP         210
GUKOSR       198
Name: count, dtype: int64


Only around 3 thousand satellites will end up with no metadata

Let's add the data:

In [104]:
# Load the categories document
df_cats = pd.read_csv(path + 'celes_dataset_concat.csv', low_memory=False)

# 1. Garantir que os IDs são strings limpas para o match
combined_df['Satcat'] = combined_df['Satcat'].astype(str).str.strip()
df_cats['NORAD_CAT_ID'] = df_cats['NORAD_CAT_ID'].astype(str).str.strip()

# 2. Lista das categorias que queres extrair (exatamente como no teu cabeçalho)
category_columns = [
    'GOES', 'NOAA', 'WEATHER', 'COMMUNICATION', 'ACTIVE_GEO', 'GEO_PROTECTED_ZONE',
    'GEO_PROTECTED_ZONE_PLUS', 'MOVERS', 'AMATEUR_RADIO', 'EUTELSAT', 'EXPERIMENTAL_COMM',
    'GLOBALSTAR', 'HULIANWANG_DIGUI', 'INTELSAT', 'IRIDIUM_NEXT', 'KUIPER', 'ONEWEB',
    'ORBCOMM', 'OTHER_COMM', 'QIANFAN', 'SATNOGS', 'SES', 'STARLINK', 'TELESAT',
    'BRIGHTEST', 'ARGOS', 'DISASTER_MONITORING', 'EARTH_RESOURCES', 'PLANET', 'SARSAT',
    'SPIRE', 'TDRSS', 'MISCELLANEOUS', 'CUBESTATS', 'MILITARY', 'OTHER', 'RADAR_CALIBRATION',
    'NAVIGATION', 'AUGMENTATION_SYSTEM', 'BEIDOU', 'GALILEO', 'GLONASS_OPERATIONAL',
    'GNSS', 'GPS_OPERATIONAL', 'NNSS', 'RUSSIAN_LEO_NAVIGATION', 'SCIENTIFIC',
    'EDUCATION', 'ENGINEERING', 'GEODETIC', 'SPACE_EARTH'
]

# 3. Criar as colunas no combined_df preenchidas com 0
for cat in category_columns:
    combined_df[cat] = 0

# 4. Mapeamento eficiente
# Vamos iterar apenas pelas categorias que realmente existem no ficheiro de entrada
present_cats = [c for c in category_columns if c in df_cats.columns]

for cat in present_cats:
    # Selecionamos os IDs onde o valor na coluna é True
    # O Pandas converte automaticamente 'True' no CSV para o booleano True do Python
    ids_verdadeiros = df_cats[df_cats[cat] == True]['NORAD_CAT_ID'].unique()
    
    # Marcamos com 1 no combined_df os satélites que pertencem a esta categoria
    combined_df.loc[combined_df['Satcat'].isin(ids_verdadeiros), cat] = 1

# Verificação final
total_encontrado = combined_df[category_columns].sum().sum()
print(f"Total de marcações de categoria realizadas: {total_encontrado}")

# Exemplo prático: Ver o ISS (25544)
if '25544' in combined_df['Satcat'].values:
    print("\nCategorias detetadas para o ISS (25544):")
    row_iss = combined_df[combined_df['Satcat'] == '25544'][category_columns]
    print(row_iss.columns[(row_iss == 1).iloc[0]].tolist())

Total de marcações de categoria realizadas: 30387

Categorias detetadas para o ISS (25544):
['COMMUNICATION', 'AMATEUR_RADIO', 'SATNOGS', 'BRIGHTEST', 'TDRSS']


### Normalize Column Names

In [105]:
print("Columns before:", combined_df.columns)

combined_df = combined_df.rename(columns={
    'Satcat': 'NORAD_CAT_ID',
    'Piece': 'OBJECT_ID',
    'Type': 'OBJECT_TYPE',
    'Name': "NAME",
    'LDate': 'LAUNCH_DATE',
    'SDate': 'SEPARATION_DATE',
    'Primary': 'PRIMARY',
    'DDate': 'DECAY_DATE',
    'Status': 'ORB_STATUS',
    'Owner': 'OWNER',
    'State': 'COUNTRY',
    'Manufacturer': 'MANUFACTURER',
    'Mass':'MASS',
    'Length': 'LENGTH',
    'Diameter': 'DIAMEETER',
    'Span':'SPAN',
    'Shape':'SHAPE',
})

print("Columns after:",combined_df.columns)

Columns before: Index(['#JCAT', 'Satcat', 'Piece', 'Type', 'Name', 'LDate', 'Parent', 'SDate',
       'Primary', 'DDate', 'Status', 'Owner', 'State', 'Manufacturer', 'Mass',
       'Length', 'Diameter', 'Span', 'Shape', 'OPS_STATUS_CODE', 'LAUNCH_SITE',
       'GOES', 'NOAA', 'WEATHER', 'COMMUNICATION', 'ACTIVE_GEO',
       'GEO_PROTECTED_ZONE', 'GEO_PROTECTED_ZONE_PLUS', 'MOVERS',
       'AMATEUR_RADIO', 'EUTELSAT', 'EXPERIMENTAL_COMM', 'GLOBALSTAR',
       'HULIANWANG_DIGUI', 'INTELSAT', 'IRIDIUM_NEXT', 'KUIPER', 'ONEWEB',
       'ORBCOMM', 'OTHER_COMM', 'QIANFAN', 'SATNOGS', 'SES', 'STARLINK',
       'TELESAT', 'BRIGHTEST', 'ARGOS', 'DISASTER_MONITORING',
       'EARTH_RESOURCES', 'PLANET', 'SARSAT', 'SPIRE', 'TDRSS',
       'MISCELLANEOUS', 'CUBESTATS', 'MILITARY', 'OTHER', 'RADAR_CALIBRATION',
       'NAVIGATION', 'AUGMENTATION_SYSTEM', 'BEIDOU', 'GALILEO',
       'GLONASS_OPERATIONAL', 'GNSS', 'GPS_OPERATIONAL', 'NNSS',
       'RUSSIAN_LEO_NAVIGATION', 'SCIENTIFIC', 'EDUCATION', 

## Clean

### Dates

We now clean the dates. Some of them had a "?" in the end, and some have the hour and others don't. So we decided to drop the time data.

In [106]:
def load_and_clean_jonathan_data(df):

    df.rename(columns={'#JCAT': 'JCAT'}, inplace=True)

    df = df[~df['JCAT'].astype(str).str.startswith('#')].copy()

    # 3. Função Interna para processar as datas
    def parse_space_dates(series):
        # Remove o '?' que indica incerteza
        clean = series.astype(str).str.replace('?', '', regex=False).str.strip()
        
        # Converte para datetime
        # O Pandas identifica automaticamente formatos como "1957 Oct 4"
        clean = clean.replace('-', np.nan)
        
        # Retorna apenas a DATA (ano-mes-dia), eliminando as horas
        # Isso uniformiza "1957 Oct 4 1933" e "1957 Oct 4"
        return pd.to_datetime(clean, errors='coerce').dt.normalize()

    # Aplicar às colunas de data
    df['SEPARATION_DATE'] = parse_space_dates(df['SEPARATION_DATE'])
    df['DECAY_DATE'] = parse_space_dates(df['DECAY_DATE'])
    df['LAUNCH_DATE'] = parse_space_dates(df['LAUNCH_DATE'])

    return df

# Exemplo de uso:
combined_df = load_and_clean_jonathan_data(combined_df)

# Verificação:
print(combined_df[['NAME', 'LAUNCH_DATE', 'SEPARATION_DATE', 'DECAY_DATE']].head())

                       NAME LAUNCH_DATE SEPARATION_DATE DECAY_DATE
0  8K71PS No. M1-10 Stage 2  1957-10-04      1957-10-04 1957-12-01
1                   1-y ISZ  1957-10-04      1957-10-04        NaT
2                   2-y ISZ  1957-11-03      1957-11-03 1958-04-14
3                Explorer I  1958-02-01      1958-02-01 1970-03-31
4                Vanguard I  1958-03-17      1958-03-17        NaT


### Mass & Length

In [114]:
# 1. Limpeza de Identificadores e Datas
# NORAD_CAT_ID como Int64 para evitar erro de NaN para Integer
combined_df['NORAD_CAT_ID'] = pd.to_numeric(combined_df['NORAD_CAT_ID'], errors='coerce').astype('Int64')

# Converter colunas de data para datetime (formato ISO)
for date_col in ['LAUNCH_DATE', 'DECAY_DATE', 'SEPARATION_DATE']:
    if date_col in combined_df.columns:
        combined_df[date_col] = pd.to_datetime(combined_df[date_col], errors='coerce')

# 2. Limpeza de Colunas Numéricas/Físicas
# Nota: Mantive o nome 'DIAMEETER' conforme está no teu log
colunas_num = ['MASS', 'LENGTH', 'DIAMEETER', 'SPAN']

for col in colunas_num:
    if col in combined_df.columns:
        # Remove espaços, converte para float, o que não for número vira NaN
        combined_df[col] = pd.to_numeric(combined_df[col], errors='coerce')

# 3. Limpeza de Strings e Categorias
# Vamos remover espaços em branco e padronizar nulos
# Owner e Country são fundamentais para a tua análise
cols_str = ['ORB_STATUS', 'OWNER', 'COUNTRY', 'OBJECT_TYPE', 'OPS_STATUS_CODE', 'LAUNCH_SITE']

for col in cols_str:
    if col in combined_df.columns:
        # strip() remove espaços; replace trata strings 'nan' como nulo real
        combined_df[col] = combined_df[col].astype(str).str.strip().replace(['nan', 'None', '-', 'unknown', 'UNKNOWN'], np.nan)

# 4. Padronização de Caixa Alta (Uppercase) para códigos
# Garante que 'pay' e 'PAY' sejam o mesmo no agrupamento
if 'OBJECT_TYPE' in combined_df.columns:
    combined_df['OBJECT_TYPE'] = combined_df['OBJECT_TYPE'].str.upper()

if 'OPS_STATUS_CODE' in combined_df.columns:
    combined_df['OPS_STATUS_CODE'] = combined_df['OPS_STATUS_CODE'].str.upper()

# 5. Opcional: Corrigir o erro de escrita da coluna Diameter (para ficar profissional no mestrado)
combined_df = combined_df.rename(columns={'DIAMEETER': 'DIAMETER'})

print("--- Limpeza de colunas concluída com sucesso ---")
print(combined_df[['NORAD_CAT_ID', 'MASS', 'COUNTRY', 'OPS_STATUS_CODE']].head())

--- Limpeza de colunas concluída com sucesso ---
   NORAD_CAT_ID    MASS COUNTRY OPS_STATUS_CODE
0             1  7790.0      SU               D
1             2    84.0      SU               D
2             3   508.0      SU               D
3             4     8.0      US               D
4             5     2.0      US             NaN


### Finish

We now have the merged dataset

In [ ]:
# =====================================================================
# SAVE AND PRINT SUMMARY
# =====================================================================

combined_df.to_csv(f'{path}/merged_dataset.csv', index=False)
print(f"Process Complete. File saved as '{path}/merged_dataset.csv'")

Process Complete. File saved as '../DATASETS_SATTELITES//merged_dataset.csv'
